# Joint + Joint Motion — fixed 50/50 late fusion

This notebook performs score-level fusion on the complete NTU60 `xsub_val` predictions. It does not train a model and does not tune alpha.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/mzuyyy/Human-action-recognition.git'
PROJECT_DIR = Path('/kaggle/working/ntu-action-recognition')
FUSION_DIR = PROJECT_DIR / 'artifacts/fusion'

if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)
os.chdir(PROJECT_DIR)
FUSION_DIR.mkdir(parents=True, exist_ok=True)
print('project:', PROJECT_DIR)


In [ ]:
# Locate complete bundles; sample alignment is verified inside the script.
REQUIRED = ('y_true.npy', 'y_score.npy', 'predictions.csv')

def complete(directory):
    return directory.is_dir() and all(
        (directory / name).is_file() for name in REQUIRED)

def input_candidates(metrics_name, expected_input):
    candidates = []
    kaggle_input = Path('/kaggle/input')
    if not kaggle_input.is_dir():
        return candidates
    for metrics_path in kaggle_input.rglob(metrics_name):
        directory = metrics_path.parent
        if not complete(directory):
            continue
        try:
            metadata = json.loads(metrics_path.read_text())
        except (json.JSONDecodeError, OSError):
            continue
        representation = str(metadata.get('input', '')).lower()
        if representation == expected_input:
            candidates.append(directory)
    return sorted(set(candidates))

joint_local = PROJECT_DIR / 'artifacts/evaluation'
motion_local = PROJECT_DIR / 'artifacts/experiments/joint_motion'
joint_candidates = ([joint_local] if complete(joint_local) else
                    input_candidates('baseline_metrics.json', 'joint'))
motion_candidates = ([motion_local] if complete(motion_local) else
                     input_candidates('metrics.json', 'joint_motion'))
if len(joint_candidates) != 1:
    raise RuntimeError(
        f'expected exactly one complete Joint bundle, found {joint_candidates}')
if len(motion_candidates) != 1:
    raise RuntimeError(
        'expected exactly one complete Joint Motion bundle, found '
        f'{motion_candidates}')
JOINT_DIR = joint_candidates[0]
MOTION_DIR = motion_candidates[0]
print('Joint bundle:', JOINT_DIR)
print('Joint Motion bundle:', MOTION_DIR)


In [ ]:
# Fixed 0.5/0.5 only. This command contains no training entry point.
command = [
    sys.executable, 'scripts/fuse_joint_motion.py',
    '--joint-dir', str(JOINT_DIR),
    '--motion-dir', str(MOTION_DIR),
    '--output-dir', str(FUSION_DIR),
]
subprocess.run(command, cwd=PROJECT_DIR, check=True)


In [ ]:
required_outputs = (
    'joint_motion_fusion_metrics.json', 'predictions.csv',
    'per_class_accuracy.csv', 'targeted_confusions.csv',
    'disagreement_analysis.json', 'confusion_matrix.png', 'report.md')
missing = [name for name in required_outputs if not (FUSION_DIR / name).is_file()]
if missing:
    raise RuntimeError(f'fusion outputs missing: {missing}')
metrics = json.loads(
    (FUSION_DIR / 'joint_motion_fusion_metrics.json').read_text())
assert metrics['num_samples'] == 16487
assert metrics['weights'] == {'joint': 0.5, 'joint_motion': 0.5}
assert metrics['alpha_tuned'] is False
print(json.dumps(metrics, indent=2))


In [ ]:
from IPython.display import Image, Markdown, display

display(Markdown((FUSION_DIR / 'report.md').read_text()))
display(Image(filename=str(FUSION_DIR / 'confusion_matrix.png')))
print('Fusion artifacts:')
for path in sorted(FUSION_DIR.iterdir()):
    if path.is_file():
        print('-', path.relative_to(PROJECT_DIR))
